In [1]:
# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Running locally (not in Colab)")

import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Set project paths
PROJECT_ROOT = '/content/drive/MyDrive/Colab_Projects/BigData/' if IN_COLAB else './'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/models', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/results', exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Colab Environment: {IN_COLAB}")

Mounted at /content/drive
Project Root: /content/drive/MyDrive/Colab_Projects/BigData/
Colab Environment: True


In [2]:
# Install required packages for Colab
import subprocess

packages_to_install = [
    'pyspark',
    'shap',
    'lime',
    'gradio',
    'tensorflow',
    'scikit-learn',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'plotly'
]

for package in packages_to_install:
    try:
        __import__(package)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("\nAll packages installed successfully!")

✓ pyspark already installed
✓ shap already installed
Installing lime...
✓ gradio already installed
✓ tensorflow already installed
Installing scikit-learn...
✓ pandas already installed
✓ numpy already installed
✓ matplotlib already installed
✓ seaborn already installed
✓ plotly already installed

All packages installed successfully!


In [ ]:
# Generate 1.2M network traffic records
np.random.seed(42)

n_records = 1_200_000  # 1.2 Million records
anomaly_ratio = 0.05   # 5% anomalies

# Generate base timestamp
start_date = datetime(2024, 1, 1)
timestamps = [start_date + timedelta(seconds=int(i * 86400 / (n_records / 365))) for i in range(n_records)]

# Generate source and destination IPs
source_ips = [f"192.168.{np.random.randint(0, 255)}.{np.random.randint(0, 255)}" for _ in range(n_records)]
dest_ips = [f"10.0.{np.random.randint(0, 255)}.{np.random.randint(0, 255)}" for _ in range(n_records)]

# Generate network features
protocols = np.random.choice(['TCP', 'UDP', 'ICMP', 'DNS'], n_records, p=[0.5, 0.3, 0.1, 0.1])
ports = np.random.choice([22, 80, 443, 3306, 5432, 9200, np.random.randint(10000, 60000)], n_records)

# Generate numerical features with anomaly injection
packet_count = np.random.exponential(100, n_records).astype(int) + 10
bytes_transferred = (packet_count * np.random.exponential(500, n_records)).astype(int)
duration_ms = (bytes_transferred / np.random.uniform(100, 1000, n_records)).astype(int)

# Inject anomalies
anomaly_indices = np.random.choice(n_records, int(n_records * anomaly_ratio), replace=False)
labels = np.zeros(n_records, dtype=int)

for idx in anomaly_indices:
    labels[idx] = 1
    packet_count[idx] = np.random.randint(5000, 50000)  # Abnormally high
    bytes_transferred[idx] = np.random.randint(100_000_000, 500_000_000)  # DDoS-like

# Create DataFrame before Spark
data_dict = {
    'timestamp': timestamps,
    'source_ip': source_ips,
    'dest_ip': dest_ips,
    'protocol': protocols,
    'port': ports,
    'packet_count': packet_count,
    'bytes_transferred': bytes_transferred,
    'duration_ms': duration_ms,
    'is_anomaly': labels
}

pandas_df = pd.DataFrame(data_dict)

print(f"Generated Dataset Summary:")
print(f"  Total Records: {len(pandas_df):,}")
print(f"  Anomalies: {pandas_df['is_anomaly'].sum():,} ({pandas_df['is_anomaly'].mean()*100:.1f}%)")
print(f"  Date Range: {pandas_df['timestamp'].min()} to {pandas_df['timestamp'].max()}")
print(f"\nDataset Shape: {pandas_df.shape}")
print(f"Memory Usage: {pandas_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nFirst 5 rows:")
print(pandas_df.head())

In [ ]:
# Introduce realistic data quality issues
from pyspark.sql.functions import when

# Create dataset with missing values and outliers
df_dirty = spark_df_partitioned.withColumn(
    'packet_count',
    when(rand() < 0.02, None).otherwise(col('packet_count'))  # 2% missing
).withColumn(
    'bytes_transferred',
    when(rand() < 0.01, None).otherwise(col('bytes_transferred'))  # 1% missing
)

print("=== DATA QUALITY ASSESSMENT ===")
print(f"Total Records: {df_dirty.count():,}")
print(f"\nNull Value Analysis:")
df_dirty.select([count(when(col(c).isNull(), c)).alias(c) for c in df_dirty.columns]).show()

In [ ]:
# ETL STEP 1: Handle Missing Values (Lazy Evaluation)
from pyspark.sql.functions import mean as spark_mean, col

df_cleaned = df_dirty

# Calculate mean for imputation (lazy)
mean_packet = df_dirty.select(spark_mean('packet_count')).collect()[0][0]
mean_bytes = df_dirty.select(spark_mean('bytes_transferred')).collect()[0][0]
mean_duration = df_dirty.select(spark_mean('duration_ms')).collect()[0][0]

# Impute with mean (lazy transformation)
df_cleaned = df_cleaned.fillna({
    'packet_count': int(mean_packet),
    'bytes_transferred': int(mean_bytes),
    'duration_ms': int(mean_duration)
})

print(f"\nAfter Imputation - Null Values:")
df_cleaned.select([count(when(col(c).isNull(), c)).alias(c) for c in df_cleaned.columns]).show()

In [ ]:
# ETL STEP 2: Outlier Detection & Capping (Lazy)
from pyspark.sql.functions import percentile_approx

# Calculate percentiles (lazy - only computed when action triggered)
q95_packets = df_cleaned.approxQuantile('packet_count', [0.95], 0.05)[0]
q95_bytes = df_cleaned.approxQuantile('bytes_transferred', [0.95], 0.05)[0]

df_cleaned = df_cleaned.withColumn(
    'packet_count',
    when(col('packet_count') > q95_packets, q95_packets).otherwise(col('packet_count'))
).withColumn(
    'bytes_transferred',
    when(col('bytes_transferred') > q95_bytes, q95_bytes).otherwise(col('bytes_transferred'))
)

print(f"95th Percentile - Packet Count: {q95_packets:,.0f}")
print(f"95th Percentile - Bytes Transferred: {q95_bytes:,.0f}")
print(f"Records after outlier capping: {df_cleaned.count():,}")

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, sum as spark_sum, avg as spark_avg, stddev
from itertools import chain

# FEATURE ENGINEERING: Create sophisticated features for ML


# Use rowsBetween instead of rangeBetween since we're working with row counts
window_per_source = Window.partitionBy('source_ip').orderBy('timestamp').rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_engineered = df_cleaned.withColumn(
    'pkt_velocity',  # Cumulative packets per source IP
    spark_sum('packet_count').over(window_per_source)
).withColumn(
    'bytes_velocity',  # Cumulative bytes per source IP
    spark_sum('bytes_transferred').over(window_per_source)
)

# 2. Protocol encoding - simplified approach
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer(inputCol='protocol', outputCol='protocol_idx')
df_engineered = indexer.fit(df_engineered).transform(df_engineered)

# 3. Time-based features
df_engineered = df_engineered.withColumn(
    'is_business_hours',
    when((col('hour') >= 9) & (col('hour') <= 17), 1).otherwise(0)
).withColumn(
    'bytes_per_packet',
    col('bytes_transferred') / (col('packet_count') + 1)  # Avoid division by zero
).withColumn(
    'packet_duration_ratio',
    col('packet_count') / (col('duration_ms') + 1)
)

# Features to use for ML
feature_columns = [
    'packet_count', 'bytes_transferred', 'duration_ms',
    'pkt_velocity', 'bytes_velocity', 'protocol_idx',
    'is_business_hours', 'bytes_per_packet', 'packet_duration_ratio'
]

print(f"\n=== ENGINEERED FEATURES ===")
print(f"Feature Columns: {feature_columns}")
print(f"Total Features: {len(feature_columns)}")
print(f"\nSample Engineered Data:")
df_engineered.select(['source_ip', 'protocol', 'packet_count', 'pkt_velocity', 'bytes_per_packet', 'is_anomaly']).show(5)


In [ ]:
# Cache engineered dataset
df_engineered.cache()
df_engineered.count()  # Trigger caching

print("ETL Pipeline Complete! Dataset ready for ML.")
print(f"Final Dataset: {df_engineered.count():,} records × {len(feature_columns)} features")

In [ ]:
# Split data appropriately (NOT collecting full data)
from pyspark.ml.feature import StandardScaler, VectorAssembler

# Assemble features (lazy)
feature_assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol='features_raw'
)

df_features = feature_assembler.transform(df_engineered)

# Standardize features
scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features',
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(df_features)
df_scaled = scaler_model.transform(df_features)

# Select relevant columns
df_ml = df_scaled.select(['features', 'is_anomaly']).cache()
df_ml.count()

print(f"ML-Ready Dataset: {df_ml.count():,} records")
print(f"Feature Vector Size: {len(feature_columns)}")
print(f"\nClass Distribution:")
df_ml.groupBy('is_anomaly').count().show()

In [ ]:
# Split into train/test WITHOUT collecting
from pyspark.sql.functions import rand

df_train, df_test = df_ml.randomSplit([0.8, 0.2], seed=42)

# Cache the splits
df_train.cache()
df_test.cache()

n_train = df_train.count()
n_test = df_test.count()

print(f"Training Set: {n_train:,} records ({n_train/(n_train+n_test)*100:.1f}%)")
print(f"Test Set: {n_test:,} records ({n_test/(n_train+n_test)*100:.1f}%)")

In [ ]:
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import time

# MODEL 1: Multilayer Perceptron (Distributed)
layers_model1 = [len(feature_columns), 128, 64, 32, 2]  # Input -> Hidden layers -> Output

mlp_classifier = MultilayerPerceptronClassifier(
    layers=layers_model1,
    labelCol='is_anomaly',  # Specify the label column name
    featuresCol='features',  # Specify the features column name
    blockSize=128,
    seed=42,
    maxIter=50,
    tol=0.0001
)

print("=== MODEL 1: MULTILAYER PERCEPTRON ===")
print(f"Architecture: {' -> '.join(map(str, layers_model1))}")
# Calculate parameters (avoids sum() shadowing from pyspark.sql.functions import *)
total_params = 0
for i in range(len(layers_model1)-1):
    total_params += (layers_model1[i] + 1) * layers_model1[i+1]
print(f"Parameters to train: {total_params:,}")
print(f"Training started...")

start_time = time.time()
mlp_model = mlp_classifier.fit(df_train)
training_time_mlp = time.time() - start_time

print(f"Training completed in {training_time_mlp:.2f} seconds")


In [ ]:
# Evaluate MODEL 1 on test set (lazy until action)
mlp_predictions = mlp_model.transform(df_test)
mlp_predictions.cache()
mlp_predictions.count()

# Calculate metrics
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator_auc = BinaryClassificationEvaluator(
    labelCol='is_anomaly',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)

auc_mlp = evaluator_auc.evaluate(mlp_predictions)

evaluator_pr = BinaryClassificationEvaluator(
    labelCol='is_anomaly',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderPR'
)

pr_mlp = evaluator_pr.evaluate(mlp_predictions)

print(f"\n=== MODEL 1 PERFORMANCE ===")
print(f"AUC Score (ROC): {auc_mlp:.4f}")
print(f"AUC Score (PR): {pr_mlp:.4f}")

In [ ]:
# Calculate Confusion Matrix for MODEL 1
from pyspark.sql.functions import expr

# Convert float predictions to class labels
mlp_predictions_labeled = mlp_predictions.withColumn(
    'prediction_class',
    when(col('prediction') >= 0.5, 1).otherwise(0)
)

# Calculate TP, TN, FP, FN without collecting full data
tp_mlp = mlp_predictions_labeled.filter((col('is_anomaly') == 1) & (col('prediction_class') == 1)).count()
tn_mlp = mlp_predictions_labeled.filter((col('is_anomaly') == 0) & (col('prediction_class') == 0)).count()
fp_mlp = mlp_predictions_labeled.filter((col('is_anomaly') == 0) & (col('prediction_class') == 1)).count()
fn_mlp = mlp_predictions_labeled.filter((col('is_anomaly') == 1) & (col('prediction_class') == 0)).count()

accuracy_mlp = (tp_mlp + tn_mlp) / (tp_mlp + tn_mlp + fp_mlp + fn_mlp)
precision_mlp = tp_mlp / (tp_mlp + fp_mlp) if (tp_mlp + fp_mlp) > 0 else 0
recall_mlp = tp_mlp / (tp_mlp + fn_mlp) if (tp_mlp + fn_mlp) > 0 else 0
f1_mlp = 2 * (precision_mlp * recall_mlp) / (precision_mlp + recall_mlp) if (precision_mlp + recall_mlp) > 0 else 0

print(f"\nConfusion Matrix - MODEL 1 (MLP):")
print(f"  TP: {tp_mlp:,} | FP: {fp_mlp:,}")
print(f"  FN: {fn_mlp:,} | TN: {tn_mlp:,}")
print(f"\nMetrics:")
print(f"  Accuracy:  {accuracy_mlp:.4f}")
print(f"  Precision: {precision_mlp:.4f}")
print(f"  Recall:    {recall_mlp:.4f}")
print(f"  F1-Score:  {f1_mlp:.4f}")

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

# MODEL 2: Random Forest (distributed decision tree ensemble)
rf_classifier = RandomForestClassifier(
    labelCol='is_anomaly',
    featuresCol='features',
    numTrees=100,
    maxDepth=15,
    minInstancesPerNode=10,
    seed=42,
    featureSubsetStrategy='sqrt',  # sqrt(n_features)
    subsamplingRate=0.8
)

print("=== MODEL 2: RANDOM FOREST (100 Trees) ===")
print(f"Max Depth: 15 (deep trees for complex patterns)")
print(f"Feature Subsets: sqrt({len(feature_columns)})")
print(f"Training started...")

start_time = time.time()
rf_model = rf_classifier.fit(df_train)
training_time_rf = time.time() - start_time

print(f"Training completed in {training_time_rf:.2f} seconds")

In [ ]:
# Evaluate MODEL 2
rf_predictions = rf_model.transform(df_test)
rf_predictions.cache()
rf_predictions.count()

auc_rf = evaluator_auc.evaluate(rf_predictions)
pr_rf = evaluator_pr.evaluate(rf_predictions)

print(f"\n=== MODEL 2 PERFORMANCE ===")
print(f"AUC Score (ROC): {auc_rf:.4f}")
print(f"AUC Score (PR): {pr_rf:.4f}")

In [ ]:
# Calculate Confusion Matrix for MODEL 2
rf_predictions_labeled = rf_predictions.withColumn(
    'prediction_class',
    when(col('prediction') >= 0.5, 1).otherwise(0)
)

tp_rf = rf_predictions_labeled.filter((col('is_anomaly') == 1) & (col('prediction_class') == 1)).count()
tn_rf = rf_predictions_labeled.filter((col('is_anomaly') == 0) & (col('prediction_class') == 0)).count()
fp_rf = rf_predictions_labeled.filter((col('is_anomaly') == 0) & (col('prediction_class') == 1)).count()
fn_rf = rf_predictions_labeled.filter((col('is_anomaly') == 1) & (col('prediction_class') == 0)).count()

accuracy_rf = (tp_rf + tn_rf) / (tp_rf + tn_rf + fp_rf + fn_rf)
precision_rf = tp_rf / (tp_rf + fp_rf) if (tp_rf + fp_rf) > 0 else 0
recall_rf = tp_rf / (tp_rf + fn_rf) if (tp_rf + fn_rf) > 0 else 0
f1_rf = 2 * (precision_rf * recall_rf) / (precision_rf + recall_rf) if (precision_rf + recall_rf) > 0 else 0

print(f"\nConfusion Matrix - MODEL 2 (Random Forest):")
print(f"  TP: {tp_rf:,} | FP: {fp_rf:,}")
print(f"  FN: {fn_rf:,} | TN: {tn_rf:,}")
print(f"\nMetrics:")
print(f"  Accuracy:  {accuracy_rf:.4f}")
print(f"  Precision: {precision_rf:.4f}")
print(f"  Recall:    {recall_rf:.4f}")
print(f"  F1-Score:  {f1_rf:.4f}")

In [ ]:
mlp_model.write().overwrite().save(f'{PROJECT_ROOT}/models/mlp_model')
rf_model.write().overwrite().save(f'{PROJECT_ROOT}/models/rf_model')
scaler_model.write().overwrite().save(f'{PROJECT_ROOT}/models/scaler_model')

In [ ]:
# Hyperparameter tuning for MODEL 1: Learning behavior comparison
print("=== HYPERPARAMETER SENSITIVITY ANALYSIS ===")
print("\nVarying Max Iterations (affects convergence):")

for max_iter in [10, 30]:
    mlp_test = MultilayerPerceptronClassifier(
        layers=layers_model1,
        labelCol='is_anomaly',
        featuresCol='features',
        maxIter=max_iter,
        seed=42
    )
    mlp_test_model = mlp_test.fit(df_train)
    mlp_test_pred = mlp_test_model.transform(df_test)
    mlp_test_auc = evaluator_auc.evaluate(mlp_test_pred)
    print(f"  Max Iterations = {max_iter:2d}: AUC = {mlp_test_auc:.4f}")

print("\nVarying Hidden Layer Size:")
for hidden_size in [32, 64, 128]:
    layers_test = [len(feature_columns), hidden_size, 32, 2]
    mlp_test = MultilayerPerceptronClassifier(
        layers=layers_test,
        labelCol='is_anomaly',
        featuresCol='features',
        maxIter=50,
        seed=42
    )
    mlp_test_model = mlp_test.fit(df_train)
    mlp_test_pred = mlp_test_model.transform(df_test)
    mlp_test_auc = evaluator_auc.evaluate(mlp_test_pred)
    print(f"  Hidden Neurons = {hidden_size:3d}: AUC = {mlp_test_auc:.4f}")

In [ ]:
import time
import pandas as pd

print("=== SCALABILITY ANALYSIS ===")
print("\nTesting pipeline execution time at different data scales...\n")

benchmark_results = []

for scale_pct in [20, 50, 100]:
    # Sample data without collecting full dataset
    df_sample = df_engineered.sample(fraction=scale_pct/100, seed=42)

    n_sampled = df_sample.count()

    # Assembly and scaling
    start = time.time()
    df_features_temp = feature_assembler.transform(df_sample)
    df_scaled_temp = scaler_model.transform(df_features_temp)
    df_ml_temp = df_scaled_temp.select(['features', 'is_anomaly']).withColumnRenamed('is_anomaly', 'label')

    df_train_temp, df_test_temp = df_ml_temp.randomSplit([0.8, 0.2], seed=42)

    # Train MLP
    mlp_temp = MultilayerPerceptronClassifier(layers=layers_model1, maxIter=30, seed=42)
    mlp_model_temp = mlp_temp.fit(df_train_temp)

    # Predict
    predictions_temp = mlp_model_temp.transform(df_test_temp)
    n_predictions = predictions_temp.count()

    elapsed = time.time() - start

    throughput = n_predictions / elapsed  # Records per second
    benchmark_results.append({
        'Data Size %': f'{scale_pct}%',
        'Records': f'{n_sampled:,}',
        'Time (sec)': f'{elapsed:.2f}',
        'Throughput (rec/sec)': f'{throughput:,.0f}'
    })

    print(f"Data Volume: {scale_pct}% ({n_sampled:,} records)")
    print(f"  Execution Time: {elapsed:.2f} seconds")
    print(f"  Throughput: {throughput:,.0f} records/second\n")

benchmark_df = pd.DataFrame(benchmark_results)
print("\nBenchmark Summary:")
print(benchmark_df.to_string(index=False))
benchmark_df.to_csv(f'{PROJECT_ROOT}/results/benchmark_analysis.csv', index=False)